# Steps to run this Streamlit dashboard in Databricks:

In [0]:
# 1. Go to "Compute" and ensure your cluster is running.
# 2. Navigate to "Apps".
# 3. Click "Create App" and select "Streamlit" as the app type.
# 4. Upload or update your app.py file.
# 5. Go to your app in the "Apps" list.
# 6. Click "Deploy".
# 7. Click the app link to view your Streamlit dashboard.

# Fully working Streamlit dashboard code

In [0]:
import os
import streamlit as st
import pandas as pd
from databricks import sql
from databricks.sdk.core import Config
import plotly.express as px
import plotly.graph_objects as go

# Validate warehouse ID is configured in app.yaml
assert os.getenv("DATABRICKS_WAREHOUSE_ID"), (
    "DATABRICKS_WAREHOUSE_ID must be set in app.yaml."
)

# Initialize Databricks configuration
cfg = Config()


def sql_query(query: str) -> pd.DataFrame:
    # Get user access token for on-behalf-of authentication
    user_token = st.context.headers.get("X-Forwarded-Access-Token")
    with sql.connect(
        server_hostname=cfg.host,
        http_path=f"/sql/1.0/warehouses/{cfg.warehouse_id}",
        access_token=user_token,
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            # Convert Arrow format to pandas DataFrame
            return cursor.fetchall_arrow().to_pandas()


def style_chart(fig, height=230):
    # Apply consistent dark theme styling to charts
    fig.update_layout(
        height=height,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="#f8fafc", size=11),
        margin=dict(l=45, r=20, t=45, b=45),
        title_font=dict(size=13, color="#f1f5f9", family="sans-serif"),
        showlegend=False,
    )
    # Style x-axis with dark theme colors
    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        tickfont=dict(color="#94a3b8"),
        title_font=dict(color="#94a3b8"),
    )
    # Style y-axis with grid and dark theme colors
    fig.update_yaxes(
        gridcolor="#334155",
        zeroline=False,
        tickfont=dict(color="#94a3b8"),
        title_font=dict(color="#94a3b8"),
    )
    return fig


def gauge_card(title, subtitle, value, max_value, color, tick_vals, suffix=""):
    # Create gauge indicator chart
    fig = go.Figure(
        go.Indicator(
            mode="gauge+number",
            value=value,
            number={"font": {"size": 24, "color": "#f8fafc"}, "suffix": f" {suffix}"},
            title={
                "text": f"{title.upper()}<br><span style='font-size:10px;color:#64748b;font-weight:normal'>{subtitle}</span>",
                "font": {"size": 11, "color": "#94a3b8", "family": "sans-serif"},
            },
            gauge={
                "axis": {
                    "range": [0, max_value],
                    "tickmode": "array",
                    "tickvals": tick_vals,
                    "tickwidth": 1.5,
                    "tickcolor": "#94a3b8",
                    "tickfont": {"color": "#94a3b8", "size": 11},
                    "ticks": "outside",
                },
                "bar": {"color": color},
                "bgcolor": "#1e293b",
                "borderwidth": 1,
                "bordercolor": "#334155",
            },
        )
    )
    # Apply layout with consistent margins
    fig.update_layout(
        height=180, margin=dict(l=35, r=35, t=50, b=15), paper_bgcolor="rgba(0,0,0,0)"
    )
    return fig


# Global Page Set Up
st.set_page_config(page_title="Weather Data Platform", layout="wide")

# Theme Stylesheet - Dark mode styling
st.markdown(
    """
    <style>
        .stApp {
            background-color: #0f172a;
            color: #f8fafc;
        }
        header[data-testid="stHeader"] {
            background: #0f172a;
        }
        .block-container {
            padding-top: 2.5rem !important;
            padding-left: 2rem;
            padding-right: 2rem;
            max-width: 1440px;
        }
        
        /* Typography metrics components */
        .kpi-title-text {
            font-size: 11px !important;
            color: #94a3b8 !important;
            font-weight: 600 !important;
            text-transform: uppercase !important;
            letter-spacing: 0.5px !important;
            margin-bottom: 2px !important;
        }
        .kpi-value-text {
            font-size: 26px !important;
            font-weight: 700 !important;
            line-height: 1.2 !important;
            margin-bottom: 2px !important;
        }
        .kpi-note-text {
            font-size: 11px !important;
            color: #64748b !important;
        }
        
        /* Warmest/Rainiest Inner Summary Alignment & Container Layout */
        .metric-subcard {
            background: #1e293b;
            padding: 10px 14px;
            border-radius: 6px;
            border-left: 4px solid #334155;
            width: 100%;
            box-sizing: border-box;
        }
        
        /* High contrast contextual value colors */
        .txt-purple { color: #c084fc !important; }
        .txt-blue { color: #38bdf8 !important; }
        .txt-red { color: #f87171 !important; }
        .txt-yellow { color: #fbbf24 !important; }
        .txt-green { color: #34d399 !important; }
        
        /* Selectbox overrides */
        div[data-testid="stSelectbox"] label {
            color: #cbd5e1 !important;
            font-weight: 600;
        }
        div[data-baseweb="select"] > div {
            background-color: #1e293b !important;
            border-color: #334155 !important;
            color: #f8fafc !important;
        }
    </style>
    """,
    unsafe_allow_html=True,
)

# Load data from gold layer tables
yearly_df = sql_query(
    "SELECT * FROM weather_catalog.weather_platform.yearly_city_summary"
)
monthly_df = sql_query(
    "SELECT * FROM weather_catalog.weather_platform.monthly_city_summary"
)
extremes_df = sql_query(
    "SELECT * FROM weather_catalog.weather_platform.weather_extremes"
)

# Clean city names for consistent filtering
yearly_df["city"] = yearly_df["city"].str.strip()
monthly_df["city"] = monthly_df["city"].str.strip()
extremes_df["city"] = extremes_df["city"].str.strip()

# Header Block
st.title("Weather Data Platform Dashboard")
st.caption(
    "Curated weather analytics from Bronze, Silver, and Gold layers using Databricks and Streamlit."
)
st.write("")

# Filtering Box Layout
with st.container(border=True):
    f1, f2 = st.columns([1, 1])

    # Get unique cities and years for filters
    cities = sorted(yearly_df["city"].unique().tolist())
    years = sorted(yearly_df["year"].unique().tolist())

    with f1:
        selected_city = st.selectbox("City", cities, format_func=lambda x: x.title())
    with f2:
        # Default to most recent year
        selected_year = st.selectbox("Year", years, index=len(years) - 1)

# Data Processing Aggregations - Filter datasets by selection
city_yearly = yearly_df[yearly_df["city"] == selected_city].copy()
city_monthly = monthly_df[
    (monthly_df["city"] == selected_city) & (monthly_df["year"] == selected_year)
].copy()
city_extremes = extremes_df[
    (extremes_df["city"] == selected_city) & (extremes_df["year"] == selected_year)
].copy()
selected_year_data = yearly_df[
    (yearly_df["city"] == selected_city) & (yearly_df["year"] == selected_year)
].copy()

# Extract key metrics for selected city and year
if not selected_year_data.empty:
    avg_temp = float(selected_year_data["avg_temperature_c"].iloc[0])
    total_rain = float(selected_year_data["total_precipitation_mm"].iloc[0])
    max_temp = float(selected_year_data["max_temperature_c"].iloc[0])
    min_temp = float(selected_year_data["min_temperature_c"].iloc[0])
else:
    # Default to zero if no data available
    avg_temp = total_rain = max_temp = min_temp = 0

# Extract highest daily rainfall with column name flexibility
highest_daily_rain = 0
if not city_extremes.empty:
    # Try multiple possible column names
    for col in [
        "highest_daily_rainfall_mm",
        "highest_daily_rainfall",
        "max_daily_rainfall",
    ]:
        if col in city_extremes.columns:
            highest_daily_rain = float(city_extremes[col].iloc[0])
            break

st.write("")

# Top KPI Row - 5 key metrics
k1, k2, k3, k4, k5 = st.columns(5)

# Average Temperature KPI
with k1:
    with st.container(border=True):
        st.markdown(
            f'<div class="kpi-title-text">Average Temperature</div><div class="kpi-value-text txt-blue">{avg_temp:.2f} °C</div><div class="kpi-note-text">{selected_city.title()} - {selected_year}</div>',
            unsafe_allow_html=True,
        )

# Total Rainfall KPI
with k2:
    with st.container(border=True):
        st.markdown(
            f'<div class="kpi-title-text">Total Rainfall</div><div class="kpi-value-text txt-red">{total_rain:.0f} mm</div><div class="kpi-note-text">Annual volume</div>',
            unsafe_allow_html=True,
        )

# Highest Temperature KPI
with k3:
    with st.container(border=True):
        st.markdown(
            f'<div class="kpi-title-text">Highest Temperature</div><div class="kpi-value-text txt-yellow">{max_temp:.2f} °C</div><div class="kpi-note-text">Yearly maximum</div>',
            unsafe_allow_html=True,
        )

# Lowest Temperature KPI
with k4:
    with st.container(border=True):
        st.markdown(
            f'<div class="kpi-title-text">Lowest Temperature</div><div class="kpi-value-text txt-green">{min_temp:.2f} °C</div><div class="kpi-note-text">Yearly minimum</div>',
            unsafe_allow_html=True,
        )

# Highest Daily Rainfall KPI
with k5:
    with st.container(border=True):
        st.markdown(
            f'<div class="kpi-title-text">Highest Daily Rainfall</div><div class="kpi-value-text txt-purple">{highest_daily_rain:.0f} mm</div><div class="kpi-note-text">Extreme event record</div>',
            unsafe_allow_html=True,
        )

st.write("")

# Second Row - Gauges and Extremes
r2c1, r2c2, r2c3, r2c4 = st.columns([1.2, 1.2, 1.1, 1.8])

with r2c1:
    # Temperature gauge
    with st.container(border=True):
        st.plotly_chart(
            gauge_card(
                "Avg Temp",
                "Normal Range: 20-35°C",
                avg_temp,
                50,
                "#38bdf8",
                [0, 10, 20, 30, 40, 50],
                suffix="°C",
            ),
            use_container_width=True,
        )

with r2c2:
    # Rainfall gauge with dynamic scale
    with st.container(border=True):
        # Calculate dynamic max value and tick intervals
        rain_max = max(6000, ((int(total_rain) // 2000) + 1) * 2000)
        rain_ticks = list(range(0, rain_max + 1, 2000))
        st.plotly_chart(
            gauge_card(
                "Annual Rainfall",
                "City Average",
                total_rain,
                rain_max,
                "#f87171",
                rain_ticks,
                suffix="mm",
            ),
            use_container_width=True,
        )

with r2c3:
    # Historical warmest and rainiest years
    warmest_row = city_yearly.loc[city_yearly["avg_temperature_c"].idxmax()]
    rainiest_row = city_yearly.loc[city_yearly["total_precipitation_mm"].idxmax()]
    with st.container(border=True):
        st.markdown(
            f"""
            <div style="height: 196px; display: flex; flex-direction: column; justify-content: space-between; padding: 0px 0 13px 0; box-sizing: border-box;">
                <div class="metric-subcard" style="border-left-color: #34d399; height: 84px; display: flex; flex-direction: column; justify-content: center;">
                    <div class="kpi-title-text">WARMEST YEAR</div>
                    <div class="kpi-value-text txt-green" style="font-size: 21px !important; margin: 0 !important; line-height: 1.1;">{int(warmest_row["year"])}</div>
                    <div class="kpi-note-text">{warmest_row["avg_temperature_c"]:.2f} °C average</div>
                </div>
                <div class="metric-subcard" style="border-left-color: #fbbf24; height: 84px; display: flex; flex-direction: column; justify-content: center;">
                    <div class="kpi-title-text">RAINIEST YEAR</div>
                    <div class="kpi-value-text txt-yellow" style="font-size: 21px !important; margin: 0 !important; line-height: 1.1;">{int(rainiest_row["year"])}</div>
                    <div class="kpi-note-text">{rainiest_row["total_precipitation_mm"]:.0f} mm volume</div>
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )
        
with r2c4:
    # Extreme weather bar chart
    with st.container(border=True):
        fig_extreme = go.Figure(
            go.Bar(
                x=["Max Temp", "Min Temp", "Daily Rain"],
                y=[max_temp, min_temp, highest_daily_rain],
                marker_color=["#f87171", "#38bdf8", "#34d399"],
                width=0.45,
            )
        )
        fig_extreme.update_layout(
            title=f"Recorded Weather Extremes - {selected_city.title()} ({selected_year})",
            xaxis_title="Weather Metric",
            yaxis_title="Value",
            yaxis=dict(
                # Dynamic tick interval based on rainfall value
                dtick=25 if highest_daily_rain <= 200 else 50,
                showgrid=True,
                gridcolor="#334155",
            ),
        )
        st.plotly_chart(style_chart(fig_extreme, height=180), use_container_width=True)

st.write("")

# Third Row - Rainfall Trend and Monthly Temperature
r3c1, r3c2 = st.columns([1.3, 1])

with r3c1:
    # Annual rainfall history bar chart
    with st.container(border=True):
        rain_trend = city_yearly.sort_values("year").copy()
        # Convert year to string for categorical x-axis
        rain_trend["year"] = rain_trend["year"].astype(str)
        fig_rain_trend = px.bar(
            rain_trend,
            x="year",
            y="total_precipitation_mm",
            title=f"Annual Rainfall Trend History - {selected_city.title()}",
            width=None,
        )
        fig_rain_trend.update_traces(
            marker_color="#a855f7", marker_line_width=0, selector=dict(type="bar")
        )
        fig_rain_trend.update_layout(
            xaxis_title="Timeline Year",
            yaxis_title="Precipitation Volume (mm)",
            bargap=0.4,
        )
        # Force categorical x-axis
        fig_rain_trend.update_xaxes(type="category")
        st.plotly_chart(
            style_chart(fig_rain_trend, height=250), use_container_width=True
        )

with r3c2:
    # Monthly temperature line chart
    with st.container(border=True):
        month_df = city_monthly.sort_values("month").copy()
        # Create month label for x-axis
        month_df["month_label"] = month_df["month"].astype(str)
        fig_month = px.line(
            month_df,
            x="month_label",
            y="avg_temperature_c",
            markers=True,
            title=f"Monthly Temperature Pattern - {selected_city.title()} {selected_year}",
        )
        fig_month.update_traces(
            line=dict(color="#38bdf8", width=3), marker=dict(size=6, color="#f87171")
        )
        fig_month.update_layout(
            xaxis_title="Month Number", yaxis_title="Temperature (°C)"
        )
        st.plotly_chart(style_chart(fig_month, height=250), use_container_width=True)

st.write("")

# Fourth Row - Temperature Trend and City Comparison
r4c1, r4c2 = st.columns([1, 1.4])

with r4c1:
    # Multi-year temperature trend
    with st.container(border=True):
        yearly_trend = city_yearly.sort_values("year").copy()
        # Convert year to string for categorical x-axis
        yearly_trend["year"] = yearly_trend["year"].astype(str)
        fig_yearly = px.line(
            yearly_trend,
            x="year",
            y="avg_temperature_c",
            markers=True,
            title="Temperature Trend (2020-2024)",
        )
        fig_yearly.update_traces(
            line=dict(color="#fbbf24", width=3), marker=dict(size=7, color="#f87171")
        )
        fig_yearly.update_layout(
            xaxis_title="Timeline Year", yaxis_title="Avg Temp (°C)"
        )
        # Force categorical x-axis
        fig_yearly.update_xaxes(type="category")
        st.plotly_chart(style_chart(fig_yearly, height=250), use_container_width=True)

with r4c2:
    # Regional rainfall comparison across all cities
    with st.container(border=True):
        # Filter by selected year and sort by rainfall amount
        rainfall_year_df = (
            yearly_df[yearly_df["year"] == selected_year]
            .copy()
            .sort_values("total_precipitation_mm", ascending=True)
        )
        # Format city names for display
        rainfall_year_df["city"] = rainfall_year_df["city"].str.title()

        # Create horizontal bar chart
        fig_rainfall = px.bar(
            rainfall_year_df,
            x="total_precipitation_mm",
            y="city",
            orientation="h",
            title=f"Regional Rainfall Comparison by City - {selected_year}",
        )
        fig_rainfall.update_traces(marker_color="#34d399")

        fig_rainfall.update_layout(
            xaxis_title="Total Precipitation (mm)",
            yaxis_title="",
            margin=dict(l=100, r=20, t=45, b=45),
            bargap=0.25,
            yaxis=dict(type="category", dtick=1),
        )
        st.plotly_chart(style_chart(fig_rainfall, height=250), use_container_width=True)
